# 08 Eval Sequence Quality

Compare MostPopular, Markov1, and Transformer next-event quality.

In [1]:
from pathlib import Path
import dataclasses
import sys
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/root/projects/bert4rec')

In [2]:
from src.eval_sequence import evaluate_sequence_quality, load_sequence_eval_config
base_config = load_sequence_eval_config(PROJECT_ROOT / "configs" / "eval_sequence.yaml")
base_config

SequenceEvalConfig(checkpoint_path='artifacts/checkpoints/checkpoint_best.pt', event_vocab_path='artifacts/vocab/event_token_vocab.json', train_config_path='configs/train.yaml', output_path='artifacts/reports/sequence_metrics.csv', manifest_path='artifacts/manifests/sequence_eval_manifest.json', input_prefixes={'train': 'data/processed/train_prefixes.parquet', 'valid': 'data/processed/valid_prefixes.parquet', 'test': 'data/processed/test_prefixes.parquet'}, k=10, batch_size=512, num_workers=0, device='auto', mixed_precision=True, dry_run_rows=None)

In [3]:
DRY_RUN = False
config = base_config
if DRY_RUN:
    config = dataclasses.replace(config, dry_run_rows=256, batch_size=64, num_workers=0, device="cpu", mixed_precision=False)
print(config)

SequenceEvalConfig(checkpoint_path='artifacts/checkpoints/checkpoint_best.pt', event_vocab_path='artifacts/vocab/event_token_vocab.json', train_config_path='configs/train.yaml', output_path='artifacts/reports/sequence_metrics.csv', manifest_path='artifacts/manifests/sequence_eval_manifest.json', input_prefixes={'train': 'data/processed/train_prefixes.parquet', 'valid': 'data/processed/valid_prefixes.parquet', 'test': 'data/processed/test_prefixes.parquet'}, k=10, batch_size=512, num_workers=0, device='auto', mixed_precision=True, dry_run_rows=None)


In [4]:
manifest = evaluate_sequence_quality(config, PROJECT_ROOT)
manifest

/root/projects/bert4rec/src/model_event_encoder.py:48: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config.n_layers)


Transformer sequence eval train:   0%|          | 0/189 [00:00<?, ?it/s]

Transformer sequence eval valid:   0%|          | 0/42 [00:00<?, ?it/s]

Transformer sequence eval test:   0%|          | 0/41 [00:00<?, ?it/s]

{'created_at': '2026-05-10T14:08:27.819229+00:00',
 'checkpoint_path': 'artifacts/checkpoints/checkpoint_best.pt',
 'dry_run_rows': None,
 'output_path': 'artifacts/reports/sequence_metrics.csv',
 'rows': 27}

In [5]:
import pandas as pd
pd.read_csv(PROJECT_ROOT / config.output_path)

,model_name,split,prefix_len,mrr_at_10,hit_at_10,num_rows
0,Markov1,test,50,0.774770,0.938855,9780
1,Markov1,test,100,0.641397,0.861278,6322
2,Markov1,test,150,0.613349,0.812956,4662
3,Markov1,train,50,0.794606,0.960389,45593
4,Markov1,train,100,0.663627,0.895535,29493
5,Markov1,train,150,0.638260,0.851467,21571
6,Markov1,valid,50,0.774300,0.940407,9867
7,Markov1,valid,100,0.638640,0.860071,6439
8,Markov1,valid,150,0.608465,0.796835,4740
9,MostPopular,test,50,0.134818,0.481493,9780
